In [1]:
# ===== Cell 1: imports, paths, quick checks =====
from pathlib import Path
import json, joblib, numpy as np, pandas as pd

# ML / DL
import xgboost as xgb
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Chemistry
from rdkit import Chem
from mordred import Calculator, descriptors

# ---- paths (adjust if your folders differ) ----
MI400_ART = Path("prep/mordred/mi400/artifacts.joblib")  # SimpleImputer, VarianceThreshold, SelectKBest
XGB_DIR   = Path("artifacts/ml/XGBOOST/MORDRED_MI400")   # model.joblib  [optional: preproc.joblib]
DMLP_DIR  = Path("artifacts/dl/ImprovedDeepMLP/MORDRED_MI400")
DMLP_SUM  = Path("results/dl/ImprovedDeepMLP/MORDRED_MI400/ensemble_summary.json")

print("[CHECK] MI400 artifacts:", MI400_ART, MI400_ART.exists())
print("[CHECK] XGB model:", XGB_DIR / "model.joblib", (XGB_DIR / "model.joblib").exists())
print("[CHECK] DMLP dir:", DMLP_DIR, DMLP_DIR.exists())
print("[CHECK] DMLP summary:", DMLP_SUM, DMLP_SUM.exists())

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch device:", DEVICE)


[CHECK] MI400 artifacts: prep\mordred\mi400\artifacts.joblib True
[CHECK] XGB model: artifacts\ml\XGBOOST\MORDRED_MI400\model.joblib True
[CHECK] DMLP dir: artifacts\dl\ImprovedDeepMLP\MORDRED_MI400 True
[CHECK] DMLP summary: results\dl\ImprovedDeepMLP\MORDRED_MI400\ensemble_summary.json True
Torch device: cuda


In [2]:
# ===== Cell 2: shim for AdvancedMolecularPreprocessor + DeepMLP classes =====
import numpy as np
import torch
import torch.nn as nn

# --- A compatible shim for the pickled preprocessing object used during training
class AdvancedMolecularPreprocessor:
    """
    Shim with the same name + API as training-time class so joblib can unpickle.
    The pickled object already contains: scaler, clip_bounds, etc.
    """
    def __init__(self, method='quantile', clip_outliers=True, outlier_std=5):
        self.method = method
        self.clip_outliers = clip_outliers
        self.outlier_std = outlier_std
        self.scaler = None
        self.feature_names = None
        self.clip_bounds = None

    def _initial_clean(self, X):
        X = np.asarray(X, dtype=np.float64)
        max_finite = np.finfo(np.float32).max / 100
        X = np.nan_to_num(X, nan=0.0, posinf=max_finite, neginf=-max_finite)
        large_mask = np.abs(X) > 1e6
        X[large_mask] = np.sign(X[large_mask]) * np.log1p(np.abs(X[large_mask]))
        return X.astype(np.float32)

    def transform(self, X):
        X_clean = self._initial_clean(X)
        if getattr(self, "clip_bounds", None) is not None:
            lo, hi = self.clip_bounds
            X_clean = np.clip(X_clean, lo, hi)
        if self.scaler is None:
            return X_clean.astype(np.float32)
        X_scaled = self.scaler.transform(X_clean)
        X_scaled = np.nan_to_num(X_scaled, nan=0.0, posinf=3.0, neginf=-3.0)
        X_scaled = np.clip(X_scaled, -10, 10)
        return X_scaled.astype(np.float32)

# --- Tabular dataset for DL inference
class TabularDataset(Dataset):
    def __init__(self, X): self.X = torch.from_numpy(np.asarray(X, np.float32))
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i]

# --- ImprovedDeepMLP (match the training architecture & names)
class AttentionBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(dim, dim // 8),
            nn.ReLU(),
            nn.Linear(dim // 8, dim),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.attention(x)

class ImprovedDeepMLP(nn.Module):
    def __init__(self, in_dim, hidden_dims=(512, 256, 128, 64), dropout=0.3, use_attention=True):
        super().__init__()
        layers = []
        if use_attention:
            layers.append(AttentionBlock(in_dim))
        prev_dim = in_dim
        for i, hidden_dim in enumerate(hidden_dims):
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.GELU() if i < len(hidden_dims)//2 else nn.ReLU())
            current_dropout = dropout * (1 - i * 0.1 / len(hidden_dims))
            layers.append(nn.Dropout(current_dropout))
            prev_dim = hidden_dim
        self.feature_extractor = nn.Sequential(*layers)
        self.output_head = nn.Sequential(
            nn.Linear(hidden_dims[-1], hidden_dims[-1] // 2),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dims[-1] // 2, 1)
        )
        # init
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)

    def forward(self, x):
        return self.output_head(self.feature_extractor(x)).squeeze(1)

def safe_torch_load(path, map_location=None):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)


In [3]:
# ===== Cell 3: load artifacts & DMLP hyperparams =====
# MI-400 pipeline pieces (fitted on training set)
mi = joblib.load(MI400_ART)          # {"imputer","vt","selector"}
imp, vt, sel = mi["imputer"], mi["vt"], mi["selector"]

# XGBoost
xgb_model   = joblib.load(XGB_DIR / "model.joblib")
xgb_preproc = XGB_DIR / "preproc.joblib"
xgb_preproc = joblib.load(xgb_preproc) if Path(xgb_preproc).exists() else None

# DeepMLP folds + threshold + hyperparams
mlp_cfg = json.loads(DMLP_SUM.read_text())
mlp_thr = float(mlp_cfg.get("optimal_threshold", 0.5))
mlp_model_paths   = sorted(DMLP_DIR.glob("fold_*[0-9].pt"))
mlp_preproc_paths = sorted(DMLP_DIR.glob("fold_*_preproc.pkl"))

# pull EXACT hyperparams used during training
MLP_HP      = mlp_cfg.get("hyperparams", {})
MLP_HIDDEN  = tuple(MLP_HP.get("hidden_dims", (512, 256, 128, 64)))
MLP_DROPOUT = float(MLP_HP.get("dropout", 0.3))
MLP_ATT     = bool(MLP_HP.get("use_attention", True))

print(f"[INFO] DeepMLP folds: {len(mlp_model_paths)} models  /  {len(mlp_preproc_paths)} preprocessors")
print(f"[INFO] DeepMLP hidden_dims={MLP_HIDDEN}, dropout={MLP_DROPOUT}, use_attention={MLP_ATT}")
print(f"[INFO] DeepMLP threshold={mlp_thr:.3f}")


[INFO] DeepMLP folds: 25 models  /  25 preprocessors
[INFO] DeepMLP hidden_dims=(768, 384, 192, 96), dropout=0.18319816212111853, use_attention=True
[INFO] DeepMLP threshold=0.560


C:\Users\deepa\anaconda3\envs\chem-serve\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\deepa\anaconda3\envs\chem-serve\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator VarianceThreshold from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\deepa\anaconda3\envs\chem-serve\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SelectKBest from version 1.5.2 when u

In [12]:
# === Cell M0 — SMILES → Mordred → MI-400 helpers ===
import numpy as np, pandas as pd, joblib
from pathlib import Path
from rdkit import Chem
from mordred import Calculator, descriptors

# 1) Load your saved MI-400 preprocessing artifacts (imputer, vt, selector)
MI400_ART = Path("prep/mordred/mi400/artifacts.joblib")
mi = joblib.load(MI400_ART)
imp, vt, sel = mi["imputer"], mi["vt"], mi["selector"]

# 2) RDKit + Mordred calculator (ignore_3D so we don't need conformers)
_mordred_calc = Calculator(descriptors, ignore_3D=True)

def smiles_to_mordred_df(smiles: str) -> pd.DataFrame:
    """Compute a single-row Mordred descriptor dataframe from a SMILES."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES string.")
    desc = _mordred_calc(mol).asdict()
    return pd.DataFrame([desc])

def to_mi400(raw_df: pd.DataFrame) -> pd.DataFrame:
    """
    Project raw Mordred descriptors to your exact MI-400 training space:
      align columns → impute → variance-threshold → SelectKBest (400).
    Returns a DataFrame with the SAME 400 columns used in training.
    """
    kept_cols = list(getattr(imp, "feature_names_in_", []))
    if not kept_cols:
        raise RuntimeError("MI-400 artifacts missing feature_names_in_. Regenerate preprocessing artifacts.")

    # align test to training columns
    X = pd.DataFrame({c: pd.to_numeric(raw_df.get(c, np.nan), errors="coerce") for c in kept_cols})

    # apply saved pipeline
    X_imp = pd.DataFrame(imp.transform(X), columns=kept_cols)
    vt_cols = list(vt.get_feature_names_out(kept_cols))
    X_vt  = pd.DataFrame(vt.transform(X_imp), columns=vt_cols)

    # SelectKBest → 400 features (preserve names)
    support = sel.get_support()
    try:
        mi_cols = list(sel.get_feature_names_out(vt_cols))
    except Exception:
        mi_cols = [c for i, c in enumerate(vt_cols) if support[i]]

    X_mi = pd.DataFrame(sel.transform(X_vt), columns=mi_cols).astype(np.float32)
    return X_mi


C:\Users\deepa\anaconda3\envs\chem-serve\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\deepa\anaconda3\envs\chem-serve\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator VarianceThreshold from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\deepa\anaconda3\envs\chem-serve\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SelectKBest from version 1.5.2 when u

In [4]:
# ===== Cell 4: SMILES -> Mordred -> MI-400 utilities =====
# 2D descriptors only
_calc = Calculator(descriptors, ignore_3D=True)

def smiles_to_mordred_df(smiles: str) -> pd.DataFrame:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES")
    desc = _calc(mol).asdict()
    return pd.DataFrame([desc])

def project_to_mi400(raw_df: pd.DataFrame) -> pd.DataFrame:
    """
    Align raw Mordred columns to the training feature list (imp.feature_names_in_),
    then apply: imputer -> variance threshold -> SelectKBest (MI-400).
    Returns a DataFrame with the EXACT MI-400 feature names in training order.
    """
    kept_cols = list(getattr(imp, "feature_names_in_", []))
    if not kept_cols:
        raise RuntimeError("MI-400 artifact missing feature_names_in_; regenerate preprocessing.")

    # align and coerce to numeric
    X = pd.DataFrame(columns=kept_cols)
    for c in kept_cols:
        X[c] = pd.to_numeric(raw_df.get(c, np.nan), errors="coerce")

    X_imp = pd.DataFrame(imp.transform(X), columns=kept_cols)
    vt_cols = list(vt.get_feature_names_out(kept_cols))
    X_vt = pd.DataFrame(vt.transform(X_imp), columns=vt_cols)

    # selector output names (sklearn >=1.1 exposes get_feature_names_out)
    try:
        mi_cols = list(sel.get_feature_names_out(vt_cols))
    except Exception:
        # fallback using support mask
        mask = sel.get_support()
        mi_cols = [c for c, keep in zip(vt_cols, mask) if keep]

    X_mi = pd.DataFrame(sel.transform(X_vt), columns=mi_cols)
    return X_mi.astype(np.float32)


In [14]:
# --- PATCH: make XGBoost ignore feature names during predict ---
import numpy as np
import pandas as pd

def predict_xgb(X_mi_df):
    # make sure we pass raw numpy (no column names)
    if isinstance(X_mi_df, pd.DataFrame):
        X = X_mi_df.values
    else:
        X = np.asarray(X_mi_df)

    # apply optional extra preproc if you saved one
    if xgb_preproc is not None:
        X = xgb_preproc.transform(X)

    # disable feature-name validation so it matches training setup
    return xgb_model.predict_proba(X, validate_features=False)[:, 1]


def predict_deepmlp(X_mi_df: pd.DataFrame) -> np.ndarray:
    """Average probabilities across all DeepMLP folds."""
    device = DEVICE
    X_mi = X_mi_df.values.astype(np.float32)
    probs_all = []

    for mp, pp in zip(mlp_model_paths, mlp_preproc_paths):
        # fold-specific scaler pipeline (AdvancedMolecularPreprocessor)
        pre = joblib.load(pp)
        Xt  = pre.transform(X_mi)

        dl = DataLoader(TabularDataset(Xt), batch_size=256, shuffle=False, num_workers=0)

        model = ImprovedDeepMLP(
            in_dim=Xt.shape[1],
            hidden_dims=MLP_HIDDEN,
            dropout=MLP_DROPOUT,
            use_attention=MLP_ATT
        ).to(device)

        state = safe_torch_load(mp, map_location=device)
        model.load_state_dict(state)
        model.eval()

        fold_probs = []
        with torch.no_grad():
            for bx in dl:
                bx = bx.to(device)
                fold_probs.append(torch.sigmoid(model(bx)).cpu().numpy())
        probs_all.append(np.concatenate(fold_probs))

    return np.mean(np.stack(probs_all, axis=0), axis=0)


In [15]:
# --- TabNet blocks & model (must match your training code) ---
import math, torch, torch.nn as nn
import torch.nn.functional as F

class Sparsemax(nn.Module):
    def forward(self, logits: torch.Tensor, dim: int = -1) -> torch.Tensor:
        z = logits
        z_sorted, _ = torch.sort(z, descending=True, dim=dim)
        z_cumsum = torch.cumsum(z_sorted, dim)
        k = torch.arange(1, z.size(dim)+1, device=z.device, dtype=z.dtype).view([1]*(z.dim()-1) + [-1])
        support = 1 + k * z_sorted > z_cumsum
        k_z = support.sum(dim=dim, keepdim=True)
        tau = (z_cumsum.gather(dim, k_z.long()-1) - 1) / k_z
        return torch.clamp(z - tau, min=0.0)

_smax = Sparsemax()

class GLUBlock(nn.Module):
    def __init__(self, in_dim, out_dim, bn=True, dropout=0.0):
        super().__init__()
        self.fc = nn.Linear(in_dim, out_dim * 2)
        self.bn = nn.BatchNorm1d(out_dim * 2) if bn else nn.Identity()
        self.dropout = nn.Dropout(dropout) if dropout>0 else nn.Identity()
    def forward(self, x):
        x = self.fc(x)
        x = self.bn(x)
        x = self.dropout(x)
        a, b = x.chunk(2, dim=-1)
        return a * torch.sigmoid(b)

class FeatureTransformer(nn.Module):
    def __init__(self, dim, n_glu=2, bn=True, dropout=0.0, shared=None):
        super().__init__()
        self.blocks = nn.ModuleList()
        if shared is None:
            self.blocks.append(GLUBlock(dim, dim, bn=bn, dropout=dropout))
        else:
            self.blocks.append(shared)
        for _ in range(n_glu-1):
            self.blocks.append(GLUBlock(dim, dim, bn=bn, dropout=dropout))
    def forward(self, x):
        out = x
        for blk in self.blocks:
            out = (out + blk(out)) * math.sqrt(0.5)
        return out

class AttentiveTransformer(nn.Module):
    def __init__(self, in_dim, n_features):
        super().__init__()
        self.fc = nn.Linear(in_dim, n_features)
    def forward(self, prior_scales, x):
        a = self.fc(x) * prior_scales
        return _smax(a, dim=-1)

class TabNet(nn.Module):
    def __init__(self, n_features: int, n_steps: int = 4, n_d: int = 64, n_a: int = 64,
                 n_glu: int = 2, relaxation: float = 1.5, bn: bool = True, dropout: float = 0.1):
        super().__init__()
        self.n_features = n_features
        self.n_steps = n_steps
        self.n_d = n_d
        self.n_a = n_a
        self.relaxation = relaxation

        self.initial_bn = nn.BatchNorm1d(n_features) if bn else nn.Identity()
        self.shared = GLUBlock(n_features, n_d + n_a, bn=bn, dropout=dropout)

        self.feature_transforms = nn.ModuleList([
            FeatureTransformer(n_d + n_a, n_glu=n_glu, bn=bn, dropout=dropout, shared=None)
            for _ in range(n_steps)
        ])
        self.attentive_transforms = nn.ModuleList([
            AttentiveTransformer(n_a, n_features) for _ in range(n_steps)
        ])
        self.decision_heads = nn.ModuleList([
            nn.Linear(n_d + n_a, n_d) for _ in range(n_steps)
        ])
        self.output = nn.Linear(n_d, 1)

    def forward(self, x):
        x = self.initial_bn(x)
        B, F = x.size()
        prior_scales = torch.ones(B, F, device=x.device)

        feat0 = self.shared(x)
        a = feat0[:, :self.n_a]
        d = feat0[:, self.n_a:self.n_a+self.n_d]

        out_agg = 0
        for step in range(self.n_steps):
            M = self.attentive_transforms[step](prior_scales, a)
            feat = self.feature_transforms[step](torch.cat([a, d], dim=-1))
            a = feat[:, :self.n_a]
            d = feat[:, self.n_a:self.n_a+self.n_d]
            out_agg = out_agg + torch.relu(self.decision_heads[step](torch.cat([a, d], dim=-1)))
            prior_scales = prior_scales * (self.relaxation - M)

        logits = self.output(out_agg)
        return logits.squeeze(1)


In [10]:
# --- TabNet artifacts & params for MORDRED_MI400 ---
import json, joblib, numpy as np
from pathlib import Path
import torch
from torch.utils.data import DataLoader, Dataset

# If your AdvancedMolecularPreprocessor is already defined in this notebook, skip this guard.
try:
    AdvancedMolecularPreprocessor  # just to check symbol exists
except NameError:
    # Paste the SAME class you used during training here if needed.
    from sklearn.preprocessing import QuantileTransformer, RobustScaler, StandardScaler
    class AdvancedMolecularPreprocessor:
        def __init__(self, method='quantile', clip_outliers=True, outlier_std=5):
            self.method = method
            self.clip_outliers = clip_outliers
            self.outlier_std = outlier_std
            if method == 'quantile':
                self.scaler = QuantileTransformer(n_quantiles=1000, output_distribution='normal',
                                                  subsample=200000, random_state=42)
            elif method == 'robust':
                self.scaler = RobustScaler()
            else:
                self.scaler = StandardScaler()
            self.clip_bounds = None
        def fit(self, X):
            Xc = self._initial_clean(X)
            if self.clip_outliers and isinstance(self.scaler, StandardScaler):
                mean, std = np.mean(Xc,0), np.std(Xc,0)
                self.clip_bounds = (mean - self.outlier_std*std, mean + self.outlier_std*std)
            self.scaler.fit(Xc); return self
        def transform(self, X):
            Xc = self._initial_clean(X)
            if self.clip_bounds is not None:
                Xc = np.clip(Xc, self.clip_bounds[0], self.clip_bounds[1])
            Xc = self.scaler.transform(Xc)
            Xc = np.nan_to_num(Xc, nan=0.0, posinf=3.0, neginf=-3.0)
            return np.clip(Xc, -10, 10).astype(np.float32)
        def _initial_clean(self, X):
            X = np.asarray(X, dtype=np.float64)
            max_finite = np.finfo(np.float32).max/100
            X = np.nan_to_num(X, nan=0.0, posinf=max_finite, neginf=-max_finite)
            big = np.abs(X) > 1e6
            X[big] = np.sign(X[big]) * np.log1p(np.abs(X[big]))
            return X.astype(np.float32)

# simple dataset for batching
class TabularDataset(Dataset):
    def __init__(self, X): self.X = torch.from_numpy(np.asarray(X, np.float32))
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i]

def safe_torch_load(path, map_location=None):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Paths
TAB_DIR   = Path("artifacts/dl/TabNet/MORDRED_MI400")    # fold_*.pt + fold_*_preproc.pkl
TAB_SUM   = Path("results/dl/TabNet/MORDRED_MI400/ensemble_summary.json")
PARAMS    = Path("results/dl/TabNet/_tuned_params.json")

# Load tuned hyperparams & threshold
TUNED = json.loads(PARAMS.read_text())
tab_hp = TUNED["MORDRED_MI400"]
tab_thr = float(json.loads(TAB_SUM.read_text()).get("optimal_threshold", 0.5))

# Discover fold files
tab_model_paths   = sorted(TAB_DIR.glob("fold_*[0-9].pt"))
tab_preproc_paths = sorted(TAB_DIR.glob("fold_*_preproc.pkl"))
assert len(tab_model_paths) == len(tab_preproc_paths) > 0, f"No TabNet folds found in {TAB_DIR}"
print(f"[TabNet] folds: {len(tab_model_paths)}  | threshold={tab_thr:.3f}")


[TabNet] folds: 15  | threshold=0.440


In [24]:
# ================================================
# TabNet + Rank-Average Ensemble (XGB, TabNet, MLP)
# Updated: uses rank-average + correct thresholds
# ================================================
import json, joblib, numpy as np, pandas as pd
from pathlib import Path
import torch
from torch.utils.data import DataLoader

# ---- safety loader for torch weights
def safe_torch_load(path, map_location=None):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)

# ---- load TabNet tuned params for MORDRED_MI400 (so tab_hp exists)
PARAMS_JSON = Path("results/dl/TabNet/_tuned_params.json")
with open(PARAMS_JSON) as f:
    _TUNED = json.load(f)
tab_hp = _TUNED["MORDRED_MI400"]  # <- used below to rebuild TabNet shells

# ---- member thresholds (kept for reference)
TAB_SUM = Path("results/dl/TabNet/MORDRED_MI400/ensemble_summary.json")
with open(TAB_SUM) as f:
    tab_thr = float(json.load(f).get("optimal_threshold", 0.5))
# You already had mlp_thr earlier in the notebook when you loaded DeepMLP summary:
#   mlp_cfg = json.loads(Path(DMLP_SUM).read_text())
#   mlp_thr = float(mlp_cfg["optimal_threshold"])
# If not present here, set a safe default:
mlp_thr = globals().get("mlp_thr", 0.56)
xgb_thr = 0.50  # if you didn’t save an XGB threshold, 0.5 is fine

# ---- PRODUCTION ENSEMBLE: rank-average (OOF tuned)
RANK_AVG_THR = 0.481  # from your OOF: rank-average ['XGBOOST','TabNet','ImprovedDeepMLP'] → thr=0.481

def _rank_average(member_probs: np.ndarray) -> np.ndarray:
    """
    member_probs: (M, N) with M members and N samples.
    If N==1 (single SMILES), fallback to simple mean (rank is degenerate).
    """
    M, N = member_probs.shape
    if N == 1:
        return member_probs.mean(axis=0)  # shape (1,)
    ranks = [pd.Series(member_probs[i]).rank(method="average").to_numpy()
             for i in range(M)]
    r = np.mean(np.stack(ranks, axis=0), axis=0)
    return (r - r.min()) / (r.max() - r.min() + 1e-9)

def predict_from_smiles(smiles: str) -> dict:
    raw = smiles_to_mordred_df(smiles)
    X400_df = to_mi400(raw)

    p_xgb = predict_xgb(X400_df.values)   # (1,)
    p_mlp = predict_deepmlp(X400_df)      # (1,)
    p_tab = predict_tabnet(X400_df)       # (1,)

    P = np.vstack([
        np.atleast_1d(float(p_xgb[0])),
        np.atleast_1d(float(p_tab[0])),
        np.atleast_1d(float(p_mlp[0])),
    ])  # (3, 1)

    # rank-average with single-item fallback
    p_rank = _rank_average(P)             # (1,)
    p_ens = float(p_rank[0])

    ensemble_thr = float(RANK_AVG_THR)    # e.g., 0.481 from your OOF
    pred = int(p_ens >= ensemble_thr)

    return {
        "probability": p_ens,
        "prediction": pred,
        "members": {
            "xgboost": float(p_xgb[0]),
            "tabnet":  float(p_tab[0]),
            "deep_mlp": float(p_mlp[0]),
        },
        "ensemble": "rank-average",
        "threshold_used": ensemble_thr,  # <-- added for backward-compat print
        "thresholds": {
            "rank_avg": ensemble_thr,
            "tabnet_indiv": float(tab_thr),
            "mlp_indiv": float(mlp_thr),
            "xgb_indiv": float(xgb_thr),
        }
    }


# --- TabNet predictor ---
def predict_tabnet(X_mi_df) -> np.ndarray:
    """
    X_mi_df: pandas DataFrame of MI-400 features (columns = your 400 names)
    Returns: (N,) numpy array of probabilities averaged across folds
    """
    probs_all = []
    for mp, pp in zip(tab_model_paths, tab_preproc_paths):
        pre = joblib.load(pp)                     # AdvancedMolecularPreprocessor (from training)
        Xt  = pre.transform(X_mi_df.values)       # transform into the same space used in training folds

        dl  = DataLoader(TabularDataset(Xt), batch_size=256, shuffle=False, num_workers=0)

        model = TabNet(
            n_features=Xt.shape[1],
            n_steps=tab_hp.get("n_steps", 4),
            n_d=tab_hp.get("n_d", 64),
            n_a=tab_hp.get("n_a", 64),
            n_glu=tab_hp.get("n_glu", 2),
            relaxation=tab_hp.get("relaxation", 1.5),
            bn=True,
            dropout=tab_hp.get("dropout", 0.1),
        ).to(DEVICE)

        state = safe_torch_load(mp, map_location=DEVICE)
        model.load_state_dict(state)
        model.eval()

        fold_probs = []
        with torch.no_grad():
            for bx in dl:
                bx = bx.to(DEVICE)
                fold_probs.append(torch.sigmoid(model(bx)).cpu().numpy())
        probs_all.append(np.concatenate(fold_probs))
    return np.mean(np.stack(probs_all, axis=0), axis=0)

# --- ENSEMBLE PREDICTOR (rank-average) ---
def predict_from_smiles(smiles: str) -> dict:
    # 1) SMILES → Mordred → MI-400 dataframe
    raw = smiles_to_mordred_df(smiles)
    X400_df = to_mi400(raw)  # DataFrame (1, 400) with the exact trained MI-400 columns

    # 2) member probabilities (scalars for a single SMILES)
    p_xgb = predict_xgb(X400_df.values)      # (1,)
    p_mlp = predict_deepmlp(X400_df)         # (1,)
    p_tab = predict_tabnet(X400_df)          # (1,)

    # 3) rank-average across members
    P = np.vstack([
        np.atleast_1d(float(p_xgb[0])),
        np.atleast_1d(float(p_tab[0])),
        np.atleast_1d(float(p_mlp[0])),
    ])  # (3, 1)
    p_rank = _rank_average(P)                # (1,)
    p_ens = float(p_rank[0])

    # 4) use the OOF-tuned threshold for rank-average
    ensemble_thr = float(RANK_AVG_THR)
    pred = int(p_ens >= ensemble_thr)

    return {
        "probability": p_ens,
        "prediction": pred,
        "members": {
            "xgboost": float(p_xgb[0]),
            "tabnet":  float(p_tab[0]),
            "deep_mlp": float(p_mlp[0]),
        },
        "ensemble": "rank-average",
        "thresholds": {
            "rank_avg": ensemble_thr,
            "tabnet_indiv": float(tab_thr),
            "mlp_indiv": float(mlp_thr),
            "xgb_indiv": float(xgb_thr),
        }
    }

# --- (optional) quick smoke test ---
# print(predict_from_smiles("CCO"))
# print(predict_from_smiles("CC(=O)O"))


In [19]:
# Ensemble decision threshold (conservative)
ENSEMBLE_THR = float(max(tab_thr, mlp_thr, xgb_thr))
print("[Thresholds] tabnet=", tab_thr, " mlp=", mlp_thr, " xgb=", xgb_thr, " -> ensemble=", ENSEMBLE_THR)



[Thresholds] tabnet= 0.44000000000000006  mlp= 0.56  xgb= 0.5  -> ensemble= 0.56


In [17]:
# 1) Make sure all three members see the *same MI-400* frame and output scalars
raw = smiles_to_mordred_df("CCO")
X400_df = to_mi400(raw)  # should be shape (1, 400)

p_xgb  = predict_xgb(X400_df.values)      # shape (1,)
p_mlp  = predict_deepmlp(X400_df)         # shape (1,)
p_tab  = predict_tabnet(X400_df)          # shape (1,)

print("Shapes:", X400_df.shape, p_xgb.shape, p_mlp.shape, p_tab.shape)
print("Member probs:", float(p_xgb[0]), float(p_mlp[0]), float(p_tab[0]))
print("Ensemble mean:", float(np.mean([p_xgb[0], p_mlp[0], p_tab[0]])))


C:\Users\deepa\anaconda3\envs\chem-serve\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator QuantileTransformer from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\deepa\anaconda3\envs\chem-serve\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator QuantileTransformer from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\deepa\anaconda3\envs\chem-serve\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator QuantileTransformer from vers

Shapes: (1, 400) (1,) (1,) (1,)
Member probs: 0.016244539991021156 0.029532887041568756 0.054739609360694885
Ensemble mean: 0.03350567817687988


In [35]:
smi = "CC1=C(SC(=N1)NC)C2=NC(=NC=C2)NC3=CC=C(C=C3)N4CCN(CC4)C(=O)C"

# quick sanity: RDKit can parse?
from rdkit import Chem
assert Chem.MolFromSmiles(smi) is not None, "Invalid SMILES (RDKit couldn't parse it)."

# run the ensemble
out = predict_from_smiles(smi)
print(out)
print(
    f"\nEnsemble prob: {out['probability']:.6f} | pred: {out['prediction']}  "
    f"| members: xgb={out['members']['xgboost']:.6f}, "
    f"tabnet={out['members'].get('tabnet', float('nan')):.6f}, "
    f"mlp={out['members']['deep_mlp']:.6f} | thr={out.get('threshold_used', out.get('thresholds', {}).get('rank_avg', float('nan'))):.3f}"
)



C:\Users\deepa\anaconda3\envs\chem-serve\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator QuantileTransformer from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\deepa\anaconda3\envs\chem-serve\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator QuantileTransformer from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\deepa\anaconda3\envs\chem-serve\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator QuantileTransformer from vers

{'probability': 0.8998237649599711, 'prediction': 1, 'members': {'xgboost': 0.947839081287384, 'tabnet': 0.9946383237838745, 'deep_mlp': 0.7569938898086548}, 'ensemble': 'rank-average', 'thresholds': {'rank_avg': 0.481, 'tabnet_indiv': 0.44000000000000006, 'mlp_indiv': 0.56, 'xgb_indiv': 0.5}}

Ensemble prob: 0.899824 | pred: 1  | members: xgb=0.947839, tabnet=0.994638, mlp=0.756994 | thr=0.481
